---
numbering: false
---

# GEE Single Composite Image

Produce a single median composite of all Landsat 8/9 surface reflectance images matching our search criteria using Google Earth Engine (GEE), and download the result to a local GeoTIFF using `xee`. Useful where no single image is good enough: e.g. where scenes are cloudy, or only partially cover the area of interest.

In [ ]:
import ee      # The main Earth Engine library
import geemap  # A Python package for interactive mapping with Google Earth Engine
import xee     # Transfer data between Earth Engine and xarray

import pandas as pd
import rioxarray as rxr
import xarray as xr

import matplotlib.pyplot as plt

from shapely.geometry import shape

Connect to Google Earth Engine. You will need to provide your own project ID (if in doubt, go to [code.earthengine.google.com](https://code.earthengine.google.com/) and see the name in the top-right corner).

In [ ]:
# =====================================
# Replace this with your own project id
my_project_name = "ee-trchudley"
# =====================================

ee.Authenticate()
ee.Initialize(project=my_project_name)

## Search for Landsat data

Set out our search criteria. All images matching these criteria will go into the composite, so if your composite is patchy or cloudy, you may have to broaden your search criteria (e.g. a longer date range, or a higher cloud threshold).

In [ ]:
# Provide a name for our area of interest (AOI)
aoi_name = "bristol"

# Provide the bounding box coordinates for the AOI in the format [min_lon, min_lat, max_lon, max_lat]
bbox = [-2.76, 51.38, -2.43, 51.53]

# Provide the date range for the search in the format "YYYY-MM-DD" (the end date is exclusive)
date_min, date_max = "2023-05-01", "2023-10-01"

# Provide a cloud cover percentage threshold for filtering the search results
max_cloud_percent = 30

# Landsat collection tier: "T1" (highest geolocation quality) or "T2" (needed for some remote areas, e.g. Antarctica, with no T1 data)
tier = "T1"

# Requested bands, as a dictionary of {GEE band name: our band name}
# I've commented out some we don't need for now, but you can uncomment them if you want to use them later
bands = {
    # "SR_B1": "coastal",  # band 1, coastal/aerosol
    "SR_B2": "blue",       # band 2, blue
    "SR_B3": "green",      # band 3, green
    "SR_B4": "red",        # band 4, red
    "SR_B5": "nir",        # band 5, near-infrared
    "SR_B6": "swir1",      # band 6, shortwave infrared 1
    "SR_B7": "swir2",      # band 7, shortwave infrared 2
}

# NDI to calculate and add as an extra band - set type_of_ndi = None to skip
type_of_ndi = "ndvi"  # name of NDI calculated, for band and filename purposes
r_high = "nir"        # relevant band name for r_high
r_low = "red"         # relevant band name for r_low

Visualise the bounding box to sanity check we have the correct region:

In [ ]:
aoi = ee.Geometry.BBox(*bbox)

map = geemap.Map()
map.addLayer(aoi, {}, "AOI")
map.centerObject(aoi, zoom=11)
map

Search for Landsat 8 and 9 images according to our search parameters:

In [ ]:
# Search for Landsat 8/9 surface reflectance imagery
image_collection = (
    ee.ImageCollection(f"LANDSAT/LC09/C02/{tier}_L2")          # Landsat 9 Collection 2, Level 2 data
    .merge(ee.ImageCollection(f"LANDSAT/LC08/C02/{tier}_L2"))  # Merged with Landsat 8 Collection 2, Level 2 data
    .filterBounds(aoi)                                         # Filter to images that intersect with the AOI
    .filterDate(date_min, date_max)                            # Filter to images within the date range
    .filter(ee.Filter.lt("CLOUD_COVER", max_cloud_percent))    # Filter to images below the cloud cover threshold
    .sort("CLOUD_COVER")                                       # Sort by cloud cover in ascending order
)

print(f"Found {image_collection.size().getInfo()} matching scene(s).")

## Calculate a median composite

Convert every image to surface reflectance, and then take the per-pixel median through the collection. Other reductions are available if you would prefer: e.g. `.mean()`, `.min()`, `.max()`. We then calculate our NDI from the composite.

In [ ]:
def preprocess(image):
    """Select bands and convert a single Landsat 8/9 Collection 2 Level 2 scene to surface reflectance"""
    return (
        image
        .select(list(bands.keys()))       # Select the requested bands
        .multiply(0.0000275).add(-0.2)    # Convert DN to surface reflectance
        .rename(list(bands.values()))     # Rename the bands
    )


composite = (
    image_collection
    .map(preprocess)  # Apply the preprocess function to every image in the collection
    .median()         # Take the per-pixel median through the collection
    .clip(aoi)        # Clip to the AOI
)

# Calculate the NDI and add it to our composite as an extra band
if type_of_ndi is not None:
    ndi = composite.normalizedDifference([r_high, r_low]).rename(type_of_ndi)
    composite = composite.addBands(ndi)

Preview the composite before downloading it (this requires an active connection to GEE):

In [ ]:
map = geemap.Map()

vis_params = {
    "bands": ["red", "green", "blue"],
    "min": 0.0,  # surface reflectance value which corresponds to black
    "max": 0.3,  # surface reflectance value which corresponds to white
}

map.addLayer(composite, vis_params, name="Landsat RGB composite")
map.centerObject(aoi, zoom=11)
map

## Download to `xarray` with `xee`

Define the pixel grid to download: the CRS and pixel size of the original Landsat data, covering our AOI. Note that GEE composites lose their native projection (defaulting to `EPSG:4326` at 1-degree resolution!), so we take the CRS and pixel size from the first original Landsat scene in the collection.

In [ ]:
# Extract the CRS and pixel size from the first original Landsat scene
projection = image_collection.first().select(0).projection().getInfo()
crs = projection["crs"]
resolution = (projection["transform"][0], projection["transform"][4])  # (x, y) pixel size: y is negative as rows run north to south

# Fit the grid to the AOI footprint
grid = xee.helpers.fit_geometry(shape(aoi.getInfo()), grid_crs=crs, grid_scale=resolution)

print("grid parameters:", grid)

Download the composite as an `xarray` dataset:

In [ ]:
ds = xr.open_dataset(
    ee.ImageCollection([composite]),  # xee can only download ImageCollections, so we wrap our image in a collection of length 1
    engine="ee",
    **grid
)

# xarray is 'lazy' by default: this is the point at which GEE performs the computation and we download the data
ds = ds.compute()

ds

Convert to a `band` dimension and add appropriate attribute names, ready to be saved as a multi-band GeoTIFF.

In [ ]:
# stack variables into a band dimension, dropping the length-1 time dimension
ds = ds.to_array(dim="band").squeeze("time", drop=True)

# add the band names as a long_name attribute to the dataset
ds.attrs["long_name"] = list(ds["band"].values)

# record the CRS
ds = ds.rio.write_crs(crs)

ds

Visualise to check the image has good coverage and quality. If it is patchy or cloudy, you can broaden your search criteria above.

In [ ]:
rgb = ds.sel(band=["red", "green", "blue"])

fig, ax = plt.subplots()
rgb.plot.imshow(ax=ax, robust=True)
ax.set_aspect("equal")

Save this under a sensible name, auto-generated from parameters:

In [ ]:
filename = f"{aoi_name}_{date_min}_{date_max}_median.tif"

print("Saving to:", filename)

ds.rio.to_raster(filename, driver="GTiff", compress="deflate")

An example of how to load the data:

In [ ]:
ds_loaded = rxr.open_rasterio(filename, masked=True).squeeze()

ds_loaded["band"] = list(ds_loaded.attrs["long_name"])

ds_loaded